# 🧪 Sesión 08 · Respuestas Semánticas y Excepciones HTTP Personalizadas

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué debe ver el cliente cuando algo falla?

---

## 🖥️ Así se verá el resultado final

```json
GET /api/libros/99
HTTP/1.1 404 Not Found
{"error":{"codigo":"NO_ENCONTRADO","mensaje":"El libro 99 no existe"}}

POST /api/libros  {"titulo": ""}
HTTP/1.1 422 Unprocessable Entity
{"error":{"codigo":"VALIDACION","mensaje":"Revisa los datos enviados"}}

Nunca:
HTTP/1.1 500 Internal Server Error
{"detail":"Traceback (most recent call last)..."}
```

_El cliente necesita saber qué corregir; tú necesitas que no se filtre la implementación._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Lanzar excepciones de negocio con códigos HTTP semánticos.
- Centralizar el formato de error en un manejador global.
- Evitar respuestas 500 y fugas de detalles internos.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Un libro existente responde 200 → HTTP 200
- [ ] Un libro inexistente responde 404 con sobre de error → HTTP 404
- [ ] Los datos inválidos responden 422 con sobre de error → HTTP 422
- [ ] Un libro válido sigue respondiendo 201 → HTTP 201

## 🧩 El reto

Implementa un manejo de errores uniforme: una excepción de negocio para libros inexistentes y un manejador para los errores de validación.

**Requisitos del contrato**

- Define `LibroNoEncontrado` y regístralo con un handler que responda 404.
- El cuerpo del 404 es {"error": {"codigo": "NO_ENCONTRADO", "mensaje": ...}}.
- Los errores de validación responden 422 con el mismo sobre y codigo VALIDACION.
- Un título vacío debe rechazarse con 422.
- La API inicia con el libro 1 (Rayuela) para que el caso feliz exista.
- GET /api/libros/1 responde 200; GET /api/libros/99 responde 404.
- Ningún caso debe producir 500.

> **Pista conceptual:** FastAPI permite registrar handlers con `@app.exception_handler(...)`. El formato de error es parte del contrato, igual que el formato de éxito.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion08.md` con el formato de error elegido.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:

%%writefile app.py
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from fastapi.exceptions import RequestValidationError

app = FastAPI(title="Manejo de errores HTTP")

# Base de datos inicial
libros = {
    1: {"id": 1, "titulo": "Rayuela", "anio": 1963}
}

# Modelo para registrar libros
class Libro(BaseModel):
    titulo: str = Field(min_length=1)
    anio: int

# Excepción personalizada
class LibroNoEncontrado(Exception):
    def __init__(self, libro_id):
        self.libro_id = libro_id

# Manejador para libros inexistentes
@app.exception_handler(LibroNoEncontrado)
async def libro_no_encontrado(request: Request, exc: LibroNoEncontrado):
    return JSONResponse(
        status_code=404,
        content={
            "error": {
                "codigo": "NO_ENCONTRADO",
                "mensaje": f"El libro {exc.libro_id} no existe"
            }
        }
    )

# Manejador para errores de validación
@app.exception_handler(RequestValidationError)
async def error_validacion(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "error": {
                "codigo": "VALIDACION",
                "mensaje": "Revisa los datos enviados"
            }
        }
    )

# Consultar un libro
@app.get("/api/libros/{libro_id}")
async def obtener_libro(libro_id: int):
    if libro_id not in libros:
        raise LibroNoEncontrado(libro_id)
    return libros[libro_id]

# Registrar un libro
@app.post("/api/libros", status_code=201)
async def crear_libro(libro: Libro):
    nuevo_id = max(libros.keys()) + 1
    nuevo_libro = {
        "id": nuevo_id,
        "titulo": libro.titulo,
        "anio": libro.anio
    }
    libros[nuevo_id] = nuevo_libro
    return nuevo_libro


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Python 3.10 o superior, FastAPI y Pydantic para crear una API. |
| **Rol** | Actuar como desarrollador especializado en manejo de errores HTTP. |
| **Especificación** | Crear una excepción para libros inexistentes y validar los datos enviados. Utilizar códigos 404 y 422 con mensajes personalizados. |
| **Output** | Código completo en app.py, con respuestas JSON uniformes y sin mostrar errores internos. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'Un libro existente responde 200',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'id': 1}},
 {'descripcion': 'Un libro inexistente responde 404 con sobre de error',
  'metodo': 'GET',
  'ruta': '/api/libros/99',
  'estado': 404,
  'json_contiene': {'error': {'codigo': 'NO_ENCONTRADO'}}},
 {'descripcion': 'Los datos inválidos responden 422 con sobre de error',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': ''},
  'estado': 422,
  'json_contiene': {'error': {'codigo': 'VALIDACION'}}},
 {'descripcion': 'Un libro válido sigue respondiendo 201',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. Un libro existente responde 200
✅ 02. Un libro inexistente responde 404 con sobre de error
✅ 03. Los datos inválidos responden 422 con sobre de error
✅ 04. Un libro válido sigue respondiendo 201
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion08.md` a `auditorias/auditoria_sesion08.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Me costó más configurar las respuestas de error 404 y 422, porque necesitaba que cada error mostrara un mensaje personalizado y el código correcto.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí utilizar un diccionario para guardar los libros y comenzar con el libro Rayuela, para facilitar las pruebas de la API.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   La verificación me ayudó a comprobar que los errores debían tener el mismo formato JSON y que los datos inválidos no debían mostrar mensajes internos del programa.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
